In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv("../datasets/track_data_final.csv")
df.head()

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,267013,False,Rihanna,90.0,68997177.0,[],5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,13,album


In [3]:
df.columns

Index(['track_id', 'track_name', 'track_number', 'track_popularity',
       'track_duration_ms', 'explicit', 'artist_name', 'artist_popularity',
       'artist_followers', 'artist_genres', 'album_id', 'album_name',
       'album_release_date', 'album_total_tracks', 'album_type'],
      dtype='object')

In [4]:
import ast

def parse_genres(val):
    """Handle both real lists and stringified lists like "['pop', 'rock']"."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else [val]
        except (ValueError, SyntaxError):
            return [val]
    return []
 
df["artist_genres"] = df["artist_genres"].apply(parse_genres)
df = df[df["artist_genres"].map(len) > 0].copy()
df = df.explode("artist_genres").reset_index(drop=True)
df.rename(columns={"artist_genres": "genre"}, inplace=True)
df["genre"] = df["genre"].str.strip()



In [5]:
# need to map the genres to a smaller set of categories, maybe using the top 20 genres and grouping the rest into "Other"
df["genre"].nunique()

423

In [6]:
from util.classify_genre import categorize_genre3

df["genre_fixed"] = df["genre"].apply(categorize_genre3)
df.head()

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,genre,album_id,album_name,album_release_date,album_total_tracks,album_type,genre_fixed
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,pop,325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation,Pop
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single,Electronic & Dance
2,7H0ya83CMmgFcOhw0UB6ow,Space Song,3,77,320466,False,Beach House,72.0,2803036.0,dream pop,194CqC2Zi0kUFEPWedb3qr,Depression Cherry,2015-08-28,9,album,Pop
3,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52,145977,False,Alok,76.0,11247155.0,brazilian bass,1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,2,single,Electronic & Dance
4,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52,145977,False,Alok,76.0,11247155.0,electronic,1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,2,single,Electronic & Dance


In [7]:
df["genre_fixed"].value_counts()

genre_fixed
Rock, Metal & Punk        1795
Pop                       1183
Hip Hop & Rap             1086
Soundtrack & Specialty     742
Country & Folk             683
Electronic & Dance         603
Latin                      436
MENA & South Asian         373
R&B, Soul & Jazz           340
Other                      284
Afro & Caribbean            73
Ambient & Chill             58
Name: count, dtype: int64

In [8]:
FEATURES = [
    "track_popularity",
    "artist_popularity",
    "artist_followers",
    "track_duration_ms",
    "album_total_tracks",
]

X = df[FEATURES]
y = df["genre_fixed"]

In [9]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

In [10]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [11]:
from sklearn.neighbors import KNeighborsClassifier

k = int(np.sqrt(len(X_train))) # KNN: sqrt of n in training set
k = k if k % 2 != 0 else k + 1
 
knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1)
knn.fit(X_train_sc, y_train)

KNeighborsClassifier(metric='euclidean', n_jobs=-1, n_neighbors=79)

In [12]:
from sklearn.metrics import classification_report

y_pred = knn.predict(X_test_sc)
 
acc = accuracy_score(y_test, y_pred)
print(f"\n Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


 Accuracy: 0.4804 (48.04%)

                        precision    recall  f1-score   support

      Afro & Caribbean       0.00      0.00      0.00        15
       Ambient & Chill       0.00      0.00      0.00        11
        Country & Folk       0.31      0.31      0.31       137
    Electronic & Dance       0.38      0.39      0.38       121
         Hip Hop & Rap       0.46      0.35      0.40       217
                 Latin       0.51      0.49      0.50        87
    MENA & South Asian       0.47      0.91      0.62        75
                 Other       0.33      0.05      0.09        57
                   Pop       0.56      0.42      0.48       237
      R&B, Soul & Jazz       0.60      0.18      0.27        68
    Rock, Metal & Punk       0.49      0.71      0.58       359
Soundtrack & Specialty       0.60      0.61      0.61       148

              accuracy                           0.48      1532
             macro avg       0.39      0.37      0.35      1532
         

c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [13]:
# Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,       # let trees grow fully
    min_samples_leaf=2,
    class_weight="balanced",  # handles class imbalance
    random_state=42,
    n_jobs=-1,
)


rf.fit(X_train_sc, y_train)
y_pred = rf.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


✅ Accuracy: 0.8133 (81.33%)

                        precision    recall  f1-score   support

      Afro & Caribbean       0.50      0.67      0.57        15
       Ambient & Chill       0.29      0.36      0.32        11
        Country & Folk       0.82      0.88      0.85       137
    Electronic & Dance       0.68      0.69      0.68       121
         Hip Hop & Rap       0.91      0.86      0.88       217
                 Latin       0.94      0.97      0.95        87
    MENA & South Asian       0.97      1.00      0.99        75
                 Other       0.26      0.37      0.31        57
                   Pop       0.88      0.81      0.85       237
      R&B, Soul & Jazz       0.58      0.62      0.60        68
    Rock, Metal & Punk       0.87      0.84      0.85       359
Soundtrack & Specialty       0.89      0.86      0.88       148

              accuracy                           0.81      1532
             macro avg       0.72      0.74      0.73      1532
        